### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="immoscout_german_house_prices",
    dataset_year="2019",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Kaggle", # original source Kaggle, download source OpenML.
    original_dataset_source_download_link="https://www.openml.org/d/43342",
    download_description="""
We download the data from OpenML (original Kaggle dataset has been deleted).

Also available at:
- https://www.kaggle.com/code/shritech1404/german-housing-price-prediction (reference only, original dataset deleted)

wget https://www.openml.org/data/download/22102167/dataset -O dataset.arff && mkdir -p local-data-warehouse/immoscout_german_house_prices && mv dataset.arff local-data-warehouse/immoscout_german_house_prices/
""",
    # References
    academic_reference_bibtex=r"""@misc{OpenML43342Dataset,
  author = {{OpenML}},
  title  = {ImmoScout24 OpenML Dataset 43342},
  year   = {2023},
  howpublished = {\url{https://www.openml.org/d/43342}},
  note   = {OpenML dataset}
}
@misc{Shritech2019GermanHousingPricePrediction,
  author = {shritech1404},
  title  = {German Housing Price Prediction},
  year   = {2019},
  howpublished = {\url{https://www.kaggle.com/code/shritech1404/german-housing-price-prediction}},
  note   = {Kaggle notebook}
}
""",
    academic_reference_bibtex_key="Shritech2019GermanHousingPricePrediction,OpenML43342Dataset",
    license="CC BY-NC-SA 4.0", # From OpenML, no idea about the original license as it was on Kaggle.
    data_tags=["IID"],
    curation_comments="""
We treat this as an IID time-independent task, because the prices is not sold prices but instead the offer price, so the snapshot here represents a task where someone would want to predict the price they should offer for their house given other houses that are currently on the market. Thus, it seems like a real IID task where someone would want to build a model to know how much they should offer their houses at in the current market (with the limitation that past trends are ignored)

- We log scale it, as is appropriate for house price prediction tasks. We do not normalize by lot or living space, as both houses with a lot of land and small living space and houses with a lot of living space and small land are interesting to predict the price of, and normalization would bias towards one of them, as both cannot be homogenised.
- We drop Free_of_Relation as it indicates a (mostly) price-independent variable; showing logistic information. Note that this column also indicates that the houses are most likely from the same time, further justifying our use case of an IID task.
- We drop all cases where the price is less than 10k (most of which have a price of 0) as we think this might be a data error, or a case where the price is put to 0 to be marked as "price on request" (which is common in house listings).
- We drop all duplicates as these are most likely repeated entries given the feature space.
- We drop all rows where the living space is less than 10 (in Germany, this would likely not be enough living space for a real house, and thus might be a data error.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="LogPrice",
    problem_type="regression",
    objective_metric_name="rmse",
)

## Preprocessing

In [ ]:
import pandas as pd
import numpy as np
import arff

with open(dataset_mold.path / "dataset.arff", "r") as f:
    data = arff.load(f)
df = pd.DataFrame(data["data"], columns=[attr[0] for attr in data["attributes"]])

# Data realistic drops
df = df[df["Price"] >=10000]
df = df[df["Living_space"] >=10]

# Normalize target
df["LogPrice"] = np.log(df["Price"])

as_cat_type = [
    "Type",
    "Furnishing_quality",
    "Condition",
    "Heating",
    "Energy_certificate",
    "Energy_certificate_type",
    "Energy_efficiency_class",
    "Garagetype",
]
df[as_cat_type] = df[as_cat_type].astype("category")

as_string_dtype = [
    "Energy_source",
    "State",
    "City",
    "Place",
]
for c in as_string_dtype:
    nan_mask = df[c].isna()
    df.loc[nan_mask, c] = np.nan
    df[c] = df[c].astype("string")

# Drop cols
df = df.drop(columns=[
    "Unnamed:_0",
    "Price",
    "Free_of_Relation",
])

# Drop duplicates ignoring the target
df = df.drop_duplicates(subset=[c for c in df.columns if c != task_mold.target_column_name])

df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print("Loaded data shape:", df.shape)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()